# 60 - GPU tests on the T4 (the tests marked `gpu`, then the rest of the suite)
Runs the repository's own tests (branch `main`; the engine code is identical to tag `m3-rc`) on a Kaggle T4 with the pinned stack (torch 2.6.0+cu124, Triton 3.2.0): first `pytest -m gpu` (the Triton kernel tests and the INT8/CUDA-graph test), then `pytest -m 'not gpu'` as a regression check on the same machine. No model download is needed (the tests build tiny models).

Run on a **GPU T4 x2** session with Internet on and Persistence = Variables and Files, then press **Run All** (about 15-30 minutes, mostly installing torch 2.6.0). Do not use *Factory reset*.

In [ ]:
import os, sys, json, time, shutil, subprocess, threading, glob

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
WORK = "/kaggle/working"
REPO = f"{WORK}/turingserve"
RESULTS = f"{WORK}/results"
RUN_FULL_SUITE = True      # after the GPU tests, also run every other test as a regression check on this stack
os.makedirs(RESULTS, exist_ok=True)
os.environ["TS_KAGGLE"] = "1"


def stream(cmd, log_name, heartbeat=120, cwd=REPO):
    """Run a command, echo its output live, keep a copy in results/<log_name>, return the exit code."""
    t0 = time.time()
    done = threading.Event()

    def beat():
        while not done.wait(heartbeat):
            print(f"  ... still running ({(time.time() - t0) / 60:.0f} min)", flush=True)

    threading.Thread(target=beat, daemon=True).start()
    p = subprocess.Popen(cmd, cwd=cwd, env=dict(os.environ, PYTHONUNBUFFERED="1"),
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    with open(f"{RESULTS}/{log_name}", "w") as f:
        for line in p.stdout:
            print(line, end="", flush=True)
            f.write(line)
    p.wait()
    done.set()
    print(f"\n[{log_name}] exit code {p.returncode}, {(time.time() - t0) / 60:.1f} min\n")
    return p.returncode


print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout)

## 1. Pin the stack

In [ ]:
def torch_version():
    r = subprocess.run([sys.executable, "-c", "import torch; print(torch.__version__)"], capture_output=True, text=True)
    return r.stdout.strip()

print("torch before:", torch_version())
if torch_version() != "2.6.0+cu124":
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchvision", "torchaudio"], capture_output=True)
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.6.0", "--index-url",
                        "https://download.pytorch.org/whl/cu124"], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-2000:])
        raise RuntimeError("torch pin failed")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.45,<5", "safetensors", "fastapi",
                    "uvicorn[standard]", "httpx", "openai", "prometheus-client", "psutil", "hypothesis", "pytest",
                    "pytest-asyncio", "pytest-timeout", "datasets", "pandas", "matplotlib", "sentencepiece"],
                   capture_output=True, text=True)
if r.returncode != 0:
    print(r.stdout[-2000:], r.stderr[-2000:])
    raise RuntimeError("dependency install failed")
print("torch after:", torch_version())

## 2. Repository (branch `main`)

In [ ]:
if not os.path.isdir(f"{REPO}/.git"):
    subprocess.run(["git", "clone", "https://github.com/Muskanpandey0304/turingserve.git", REPO], check=True)
REF = "main"      # the engine package is identical to the benchmarked tag m3-rc; main also carries the current tests
subprocess.run(["git", "fetch", "--tags", "--force", "origin"], cwd=REPO, check=True, capture_output=True)
subprocess.run(["git", "checkout", "-f", REF], cwd=REPO, check=True, capture_output=True)
subprocess.run(["git", "reset", "--hard", f"origin/{REF}"], cwd=REPO, check=True, capture_output=True)
print("repo at:", subprocess.run(["git", "log", "-1", "--oneline"], cwd=REPO, capture_output=True, text=True).stdout.strip())
# If the fixed test has not reached GitHub yet, apply the same two-line fix here so this run tests the fixed case
# (n=31 needs 462 KV blocks and sequences below the 512-token model limit). A no-op once the fix is pushed.
tp = f"{REPO}/tests/correctness/test_int8_graphs.py"
t = open(tp).read()
if "    nb = 400
" in t and "        L = 5 + 17 * i
" in t:
    open(tp, "w").write(t.replace("    nb = 400
", "    nb = 600
").replace("        L = 5 + 17 * i
", "        L = 5 + 15 * i
"))
    print("applied the local test fix (it is not on GitHub yet)")
else:
    print("the test fix is already in the repository")
print(subprocess.run(["git", "status", "--short"], cwd=REPO, capture_output=True, text=True).stdout or "(working tree clean)")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], cwd=REPO, capture_output=True)
r = subprocess.run([sys.executable, "scripts/check_versions.py"], cwd=REPO, capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-800:])
open(f"{RESULTS}/versions.txt", "w").write(r.stdout)

## 3. GPU tests
Every test is listed with its result (`-rA`).

In [ ]:
gpu_rc = stream([sys.executable, "-m", "pytest", "-m", "gpu", "-v", "-rA", "--timeout=600", "-p", "no:cacheprovider",
                 f"--junitxml={RESULTS}/pytest_gpu.xml"], "pytest_gpu.txt")

## 4. Rest of the suite
Unit, correctness and end-to-end tests (tiny models, CPU). Set `RUN_FULL_SUITE = False` in the first cell to skip.

In [ ]:
full_rc = None
if RUN_FULL_SUITE:
    full_rc = stream([sys.executable, "-m", "pytest", "-m", "not gpu", "-q", "-rfEs", "--timeout=600", "-p", "no:cacheprovider",
                      f"--junitxml={RESULTS}/pytest_rest.xml"], "pytest_rest.txt", heartbeat=180)
else:
    print("full suite skipped")

## 5. Summary and download
Download **`/kaggle/working/gpu_tests_bundle.zip`**, then stop the session.

In [ ]:
import xml.etree.ElementTree as ET


def summarize(path):
    if not os.path.exists(path):
        return None
    root = ET.parse(path).getroot()
    suites = [root] if root.tag == "testsuite" else list(root)
    tot = dict(tests=0, failures=0, errors=0, skipped=0)
    bad, skipped = [], []
    for s in suites:
        for k in tot:
            tot[k] += int(s.get(k, 0))
        for tc in s.iter("testcase"):
            name = f"{tc.get('classname')}::{tc.get('name')}"
            if tc.find("failure") is not None or tc.find("error") is not None:
                bad.append(name)
            elif tc.find("skipped") is not None:
                skipped.append((name, tc.find("skipped").get("message", "")[:80]))
    tot["passed"] = tot["tests"] - tot["failures"] - tot["errors"] - tot["skipped"]
    return tot, bad, skipped


for title, f, rc in (("GPU tests (pytest -m gpu)", "pytest_gpu.xml", gpu_rc), ("All other tests (pytest -m 'not gpu')", "pytest_rest.xml", full_rc)):
    r = summarize(f"{RESULTS}/{f}")
    print(f"== {title}: exit code {rc}")
    if r is None:
        print("   not run")
        continue
    tot, bad, skipped = r
    print("  ", tot)
    for b in bad:
        print("   FAILED:", b)
    if skipped:
        print(f"   skipped ({len(skipped)}):", sorted({m for _, m in skipped})[:4])

bundle = f"{WORK}/gpu_tests_bundle"
shutil.rmtree(bundle, ignore_errors=True)
os.makedirs(bundle)
for f in glob.glob(f"{RESULTS}/*"):
    if os.path.isfile(f):
        shutil.copy(f, bundle)
shutil.make_archive(bundle, "zip", bundle)
print("\nDownload:", bundle + ".zip", f"({os.path.getsize(bundle + '.zip') / 1e6:.2f} MB)")